# This is a tutorial for doing the pileup analysis

## Imports

In [1]:
%load_ext autoreload
%autoreload 1
%aimport utility.functions
%aimport src.analysis
%aimport src.dataset
%aimport src.plots
%aimport src.simulation
import numpy as np
import src.analysis as an
import src.dataset as ds
import torch
from scipy.stats import norm
from utility.double_beta_spectrum import pdf_ratio2b
BASE = "../"
channels = [3, 5, 9, 11, 13, 15, 17, 19]
channel_ID = [5, 9, 11, 10, 4, 2, 12, 3]
StdCuts = [0.0003, 0.0004, 0.0008, 0.0006, 0.0012, 0.0007, 0.00035, 0.0025]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Choose analysis parameters

In [ ]:
meas_name = "000813_20230628T161508"
channel = 5
acceptance = 0.9
N_sigma = norm.ppf(1 - (1 - acceptance) * 100 / 100) #number of sigma corresponding to the desired acceptance
signal_amp = ds.get_amp_Q_val(channel) #expected amplitude of the signal at the Q-value
#define the time and ratio grid for the analysis
t_min, t_max, N_t = 0, 8e-4, 100
r_min, r_max, N_r = 0., .5, 100
#get the corresponding ratio distribution 
ratio_distribution = pdf_ratio2b(np.linspace(r_min, r_max, N_r))
ratio_distribution /= np.mean(ratio_distribution)

window_size = 4096 #size of the window used for the analysis, should be the same as the one used for the NPS computation
n_trials=1000 #number of trials for the optimization of the filters

## Load the meanpulse and Noise power spectrum
if they are not already compute, look build_meanpulses.py and build_NPS.py to compute them

In [ ]:
#load the meanpulse
meanpulse = np.load(f"{BASE}/outputs/meanpulses_build/channel{channel}_meanpulse_mean.npy")
if len(meanpulse) > window_size:
    meanpulse = meanpulse[np.argmax(meanpulse) - window_size//2 : np.argmax(meanpulse) + window_size//2]
#load the NPS
nps = np.load(f"{BASE}/outputs/NPS_study/NPS_channel{channel}_win{window_size}_{meas_name}.npy")
nps *= (8. / 3.)  # Adjust for Hann window effect

## Compute the optimal filter

In [ ]:
S, w, H_unit = an.compute_H(meanpulse, nps, np.hanning)

## Convert the relevant arrays to torch tensors and move them to the appropriate device (CPU or GPU)

In [ ]:
S_torch = torch.tensor(S, dtype=torch.cfloat, device=device)
H_unit_torch = torch.tensor(H_unit, dtype=torch.cfloat, device=device)
w_torch = torch.tensor(w, dtype=torch.cfloat, device=device)
nps_torch = torch.tensor(nps, dtype=torch.cfloat, device=device)
signal_amp_torch = torch.tensor(signal_amp, dtype=torch.float32, device=device)
t_torch = torch.linspace(t_min, t_max, N_t, dtype=torch.cfloat, device=device)
r_torch = torch.linspace(r_min, r_max, N_r, dtype=torch.cfloat, device=device)
ratio_distribution_torch = torch.tensor(ratio_distribution, dtype=torch.cfloat, device=device)

## Optimize the filters for minimal misidentification rate (J_value)

In [ ]:
f1_opt, f2_opt, J_values = an.optimize_filters(S_torch, H_unit_torch, w_torch, t_torch, r_torch, nps_torch,
                                                       signal_amp_torch, ratio_distribution_torch, N_sigma=N_sigma,
                                                       n_trials=n_trials, verbose=False)

f1_t = torch.tensor(f1_opt, dtype=torch.float32, device=device)
f2_t = torch.tensor(f2_opt, dtype=torch.float32, device=device)

## Save the results

In [ ]:
np.save(f"{BASE}/outputs/training_Js/channel_{channel}/J_{int(acceptance * 100):d}", np.array(J_values))
np.save(f"{BASE}/outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance * 100):d}",np.array([f1_opt, f2_opt]))

# Compute the Background Index using injected pulses

## Load the dataset
path_pos_single and path_pos_pileup are the paths to the position files for the single pulse and pileup datasets, respectively. They contain the relevent informations for the injection (position in the raw file to inject pulse , amplitudes, delay, smearing,...). Those data are created by montecarlo from IJClab code.

In [ ]:
#load the pulse to be injected
meanpulse_inj = np.fromfile(f"{BASE}/outputs/meanpulses_build/pos_pulse_channel{channel}.bin", dtype=np.float32)
#load the single pulse dataset
dataset_single_name = "pup_n1-d0_000813_20230628T161508"
file_path_single, path_pos_single = ds.find_files(meas_name, dataset_single_name, channel,
                                                  specific_subdir="RUN9_pulse_injected_new/")
data_single = ds.CachedBinaryDataset_withgenerated(file_path_single, path_pos_single, window_size,
                                                   pulse=meanpulse_inj,
                                                   n_windows=15999, win_shift=0)
#load the pileup pulse dataset
dataset_pileup_name = "pup_n1-d8_000813_20230628T161508"
file_path_pileup, path_pos_pileup = ds.find_files(meas_name, dataset_pileup_name, channel,
                                                  specific_subdir="RUN9_pulse_injected_new/")
data_pileup = ds.CachedBinaryDataset_withgenerated(file_path_pileup, path_pos_pileup, window_size,
                                                   pulse=meanpulse_inj,
                                                   n_windows=15999, win_shift=0)

## Compute the Background Index

In [ ]:
BI, rp, sigma_rp, sigma_BI = an.compute_BI_torch(
            data_pileup,
            data_single,
            acceptance,
            H_unit_torch,
            f1_t,
            f2_t,
            window_fct = np.hanning,
            n_deriv = 0,
            compute_uncertainty = True,
            batch_size = 2048,
            use_loader = False
        )